BƯỚC 1: KHỞI TẠO & LOAD DỮ LIỆU

In [ ]:
# [BƯỚC 1] Import thư viện & Load dữ liệu
import pandas as pd
import numpy as np
import re
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.pipeline import Pipeline

try:
    print(">>> Đang đọc file CSV...")
    df = pd.read_csv('MBTI 500.csv')
    print(f"✅ Đã tải ban đầu: {len(df)} dòng.")

    # --- XỬ LÝ LỖI NAN (RỖNG) ---
    initial_len = len(df)
    # Xóa dòng nếu cột 'type' hoặc 'posts' bị rỗng
    df.dropna(subset=['type', 'posts'], inplace=True)

    # Ép kiểu cột 'type' về dạng chuỗi (String) để tránh lỗi float
    df['type'] = df['type'].astype(str)

    print(f"🧹 Đã dọn dẹp {initial_len - len(df)} dòng bị lỗi/rỗng.")
    print(f"✅ Dữ liệu sạch sẵn sàng: {len(df)} dòng.")
    # ---------------------------

except FileNotFoundError:
    print("❌ LỖI: Không tìm thấy file 'MBTI 500.csv'. Nhớ upload file lên cột bên trái nha!")

>>> Đang đọc file CSV...
✅ Đã tải ban đầu: 106067 dòng.
🧹 Đã dọn dẹp 0 dòng bị lỗi/rỗng.
✅ Dữ liệu sạch sẵn sàng: 106067 dòng.


BƯỚC 2: TIỀN XỬ LÝ (LÀM SẠCH)

In [ ]:
# [BƯỚC 2] Hàm làm sạch dữ liệu (Regex)
def basic_clean(text):
    # Chuyển thành chuỗi và viết thường
    text = str(text).lower()

    # Xóa URL (link web)
    text = re.sub(r'http\S+|www\.\S+', '', text)

    # Xóa 16 từ khóa MBTI (Chống ăn gian)
    mbti_types = ['infj', 'intp', 'infp', 'entp', 'intj', 'istj', 'entj', 'istp',
                  'enfj', 'enfp', 'isfj', 'esfj', 'isfp', 'estj', 'estp', 'esfp']
    pattern = r'\b(?:' + '|'.join(mbti_types) + r')\b'
    text = re.sub(pattern, '', text)

    # Chỉ giữ lại chữ cái và số (Xóa emoji, ký tự lạ)
    text = re.sub(r'[^a-z0-9\s]', '', text)

    return text.strip()

print("⏳ Đang làm sạch dữ liệu (Chờ khoảng 30s)...")
df['clean_text'] = df['posts'].apply(basic_clean)
print("✅ Đã làm sạch xong! Sẵn sàng để Train.")
print("-" * 30)
print("Ví dụ 1 dòng sau khi clean:", df['clean_text'].iloc[0][:100], "...")

⏳ Đang làm sạch dữ liệu (Chờ khoảng 30s)...
✅ Đã làm sạch xong! Sẵn sàng để Train.
------------------------------
Ví dụ 1 dòng sau khi clean: know  tool use interaction people excuse antisocial truly enlighten mastermind know would count pet  ...


BƯỚC 3: HUẤN LUYỆN (TRAINING LOOP)

In [ ]:
# [BƯỚC 3] Vòng lặp Training & Đánh giá
axes = {
    'IE': ('I', 'E'),
    'NS': ('N', 'S'),
    'TF': ('F', 'T'),
    'JP': ('J', 'P')
}

results = [] # Nơi lưu điểm số

print("🚀 BẮT ĐẦU HUẤN LUYỆN MODEL BASELINE...")

for axis, (char0, char1) in axes.items():
    print(f"\n🔹 Đang xử lý trục: {axis} ({char0} vs {char1})")

    # 1. Tạo nhãn (0 hoặc 1)
    # Ví dụ trục IE: Nếu có chữ 'E' trong type thì là 1, ngược lại là 0
    df['label'] = df['type'].apply(lambda x: 1 if char1 in x else 0)

    # 2. Chia tập Train/Test (80/20)
    X_train, X_test, y_train, y_test = train_test_split(
        df['clean_text'], df['label'], test_size=0.2, random_state=42, stratify=df['label']
    )

    # 3. Tạo Pipeline: TF-IDF -> Logistic Regression
    pipeline = Pipeline([
        ('tfidf', TfidfVectorizer(max_features=10000, stop_words='english')),
        ('clf', LogisticRegression(solver='liblinear', class_weight='balanced'))
    ])

    # 4. Train
    pipeline.fit(X_train, y_train)

    # 5. Test & Ghi điểm
    y_pred = pipeline.predict(X_test)
    acc = accuracy_score(y_test, y_pred)

    print(f"   -> Độ chính xác (Accuracy): {acc:.4f}")
    results.append({'Axis': axis, 'Baseline Accuracy': acc})

print("\n✅ Đã chạy xong cả 4 trục!")

🚀 BẮT ĐẦU HUẤN LUYỆN MODEL BASELINE...

🔹 Đang xử lý trục: IE (I vs E)
   -> Độ chính xác (Accuracy): 0.8486

🔹 Đang xử lý trục: NS (N vs S)
   -> Độ chính xác (Accuracy): 0.9129

🔹 Đang xử lý trục: TF (F vs T)
   -> Độ chính xác (Accuracy): 0.9006

🔹 Đang xử lý trục: JP (J vs P)
   -> Độ chính xác (Accuracy): 0.8065

✅ Đã chạy xong cả 4 trục!


BƯỚC 4: TỔNG KẾT KẾT QUẢ

In [ ]:
# [BƯỚC 4] BÁO CÁO CHI TIẾT ĐẦY ĐỦ (Cập nhật: Thêm Weighted F1 & Macro F1)
from sklearn.metrics import classification_report, f1_score, accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
import pandas as pd

print("📊 ĐANG TỔNG HỢP BÁO CÁO CHI TIẾT (ACCURACY, MACRO F1 & WEIGHTED F1)...")
print("(Quá trình này sẽ mất khoảng 1-2 phút để quét lại toàn bộ)")
print("=" * 80)

detailed_results = []
axes_list = {'IE': ('I', 'E'), 'NS': ('N', 'S'), 'TF': ('F', 'T'), 'JP': ('J', 'P')}

for axis, (char0, char1) in axes_list.items():
    # 1. Tái tạo dữ liệu cho trục này
    # Giả sử df đã được load và có cột 'clean_text' và 'type'
    df['label'] = df['type'].apply(lambda x: 1 if char1 in x else 0)

    # Chia tập train/test (Stratify để giữ tỷ lệ nhãn)
    X_train, X_test, y_train, y_test = train_test_split(
        df['clean_text'], df['label'], test_size=0.2, random_state=42, stratify=df['label']
    )

    # 2. Train nhanh lại
    pipeline = Pipeline([
        ('tfidf', TfidfVectorizer(max_features=10000, stop_words='english')),
        ('clf', LogisticRegression(solver='liblinear', class_weight='balanced'))
    ])
    pipeline.fit(X_train, y_train)

    # 3. Lấy dự đoán
    y_pred = pipeline.predict(X_test)

    # 4. Tính toán các chỉ số
    acc = accuracy_score(y_test, y_pred)

    # F1-Score cho lớp Positive (Binary - dùng để tham khảo nhanh)
    f1_binary = f1_score(y_test, y_pred, average='binary')

    # Macro F1 (Trung bình cộng không trọng số - Quan trọng cho MF1 trong bảng)
    mf1 = f1_score(y_test, y_pred, average='macro')

    # Weighted F1 (Trung bình cộng có trọng số - Quan trọng cho WF1 trong bảng)
    wf1 = f1_score(y_test, y_pred, average='weighted')

    # Lưu kết quả
    detailed_results.append({
        'Trục': axis,
        'Accuracy': acc,
        'Macro F1': mf1,
        'Weighted F1': wf1
    })

    # 5. In ngay báo cáo ra màn hình
    print(f"\n🔹 KẾT QUẢ CHI TIẾT TRỤC: {axis} ({char0} vs {char1})")
    print(f"   - Accuracy (Độ chính xác):     {acc:.4f}")
    print(f"   - Macro F1 (Độ cân bằng MF1):  {mf1:.4f}")
    print(f"   - Weighted F1 (Tổng quan WF1): {wf1:.4f} <--- Chỉ số mới thêm")
    print("-" * 50)
    print(classification_report(y_test, y_pred, target_names=[char0, char1]))
    print("=" * 80)

# Tổng kết lại bảng đẹp
print("\n🏆 TỔNG HỢP KẾT QUẢ CUỐI CÙNG")
final_df = pd.DataFrame(detailed_results)
# Định dạng hiển thị số thực 4 chữ số thập phân
pd.options.display.float_format = '{:.4f}'.format
print(final_df)

📊 ĐANG TỔNG HỢP BÁO CÁO CHI TIẾT (ACCURACY, MACRO F1 & WEIGHTED F1)...
(Quá trình này sẽ mất khoảng 1-2 phút để quét lại toàn bộ)

🔹 KẾT QUẢ CHI TIẾT TRỤC: IE (I vs E)
   - Accuracy (Độ chính xác):     0.8486
   - Macro F1 (Độ cân bằng MF1):  0.8080
   - Weighted F1 (Tổng quan WF1): 0.8541 <--- Chỉ số mới thêm
--------------------------------------------------
              precision    recall  f1-score   support

           I       0.94      0.86      0.90     16136
           E       0.65      0.81      0.72      5078

    accuracy                           0.85     21214
   macro avg       0.79      0.84      0.81     21214
weighted avg       0.87      0.85      0.85     21214


🔹 KẾT QUẢ CHI TIẾT TRỤC: NS (N vs S)
   - Accuracy (Độ chính xác):     0.9129
   - Macro F1 (Độ cân bằng MF1):  0.7864
   - Weighted F1 (Tổng quan WF1): 0.9223 <--- Chỉ số mới thêm
--------------------------------------------------
              precision    recall  f1-score   support

           N       0.9

[BƯỚC 5] CODE LƯU MODEL

In [ ]:
import joblib
import os

print("💾 ĐANG TIẾN HÀNH LƯU 4 MODEL CHO 4 TRỤC...")
print("=" * 60)

# Định nghĩa lại trục (để chắc chắn)
axes_list = {'IE': ('I', 'E'), 'NS': ('N', 'S'), 'TF': ('F', 'T'), 'JP': ('J', 'P')}

# Tạo thư mục chứa model cho gọn (nếu chưa có)
if not os.path.exists('saved_models'):
    os.makedirs('saved_models')

for axis, (char0, char1) in axes_list.items():
    print(f"⚙️ Đang đóng gói model trục {axis}...")

    # 1. Tái tạo dữ liệu & Nhãn
    df['label'] = df['type'].apply(lambda x: 1 if char1 in x else 0)

    # 2. Chia tập dữ liệu (Stratify để đảm bảo chất lượng)
    X_train, X_test, y_train, y_test = train_test_split(
        df['clean_text'], df['label'], test_size=0.2, random_state=42, stratify=df['label']
    )

    # 3. Tạo Pipeline (Gồm cả TF-IDF và LogisticRegression)

    final_pipeline = Pipeline([
        ('tfidf', TfidfVectorizer(max_features=10000, stop_words='english')),
        ('clf', LogisticRegression(solver='liblinear', class_weight='balanced', random_state=42))
    ])

    # 4. Train trên tập dữ liệu
    final_pipeline.fit(X_train, y_train)

    # 5. LƯU FILE (Quan trọng nhất)
    # Tên file sẽ là: pipeline_IE.pkl, pipeline_NS.pkl,...
    filename = f'saved_models/pipeline_{axis}.pkl'
    joblib.dump(final_pipeline, filename)

    print(f"   ✅ Đã lưu thành công: {filename}")

print("=" * 60)
print("🎉 HOÀN TẤT! GIỜ TẢI FILE VỀ.")

# --- TỰ ĐỘNG TẢI FILE VỀ MÁY TÍNH ---
from google.colab import files
import time

print("\n⬇️ Đang gửi lệnh tải file xuống trình duyệt...")
try:
    files.download('saved_models/pipeline_IE.pkl')
    time.sleep(1)
    files.download('saved_models/pipeline_NS.pkl')
    time.sleep(1)
    files.download('saved_models/pipeline_TF.pkl')
    time.sleep(1)
    files.download('saved_models/pipeline_JP.pkl')
    print("✅ Đã gửi lệnh tải 4 file. Hãy kiểm tra thư mục Download của máy tính!")
except Exception as e:
    print(f"⚠️ Lỗi tự động tải: {e}")

💾 ĐANG TIẾN HÀNH LƯU 4 MODEL CHO 4 TRỤC...
⚙️ Đang đóng gói model trục IE...
   ✅ Đã lưu thành công: saved_models/pipeline_IE.pkl
⚙️ Đang đóng gói model trục NS...
   ✅ Đã lưu thành công: saved_models/pipeline_NS.pkl
⚙️ Đang đóng gói model trục TF...
   ✅ Đã lưu thành công: saved_models/pipeline_TF.pkl
⚙️ Đang đóng gói model trục JP...
   ✅ Đã lưu thành công: saved_models/pipeline_JP.pkl
🎉 HOÀN TẤT! GIỜ ÔNG TẢI FILE VỀ GỬI CHO BẠN.

⬇️ Đang gửi lệnh tải file xuống trình duyệt...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

✅ Đã gửi lệnh tải 4 file. Hãy kiểm tra thư mục Download của máy tính!


# [BƯỚC 6] TÌM KIẾM PERFECT SAMPLES ĐỂ DEMO


In [ ]:
# [BƯỚC 6] TÌM KIẾM PERFECT SAMPLES ĐỂ DEMO
print("\n💎 ĐANG TÌM CÁC MẪU ĐÚNG TUYỆT ĐỐI (4/4)...")

# 1. Chia tập Train/Test CỐ ĐỊNH (Không chia trong vòng lặp nữa)
# Lưu ý: Bỏ stratify ở đây để đơn giản hóa, hoặc stratify theo 'type' gốc nếu dữ liệu đủ lớn
train_df, test_df = train_test_split(df, test_size=0.2, random_state=42)

# Tạo một cột để lưu kết quả dự đoán gộp (Ví dụ: "INTJ")
test_df = test_df.copy() # Tránh lỗi SettingWithCopy
test_df['predicted_type'] = ""

axes_list = {'IE': ('I', 'E'), 'NS': ('N', 'S'), 'TF': ('F', 'T'), 'JP': ('J', 'P')}

# 2. Chạy lại vòng lặp training trên tập cố định này
for axis, (char0, char1) in axes_list.items():
    print(f" -> Đang train lại & dự đoán trục {axis}...")

    # Tạo nhãn cho tập Train cố định
    y_train_axis = train_df['type'].apply(lambda x: 1 if char1 in x else 0)

    # Pipeline huấn luyện
    pipeline = Pipeline([
        ('tfidf', TfidfVectorizer(max_features=10000, stop_words='english')),
        ('clf', LogisticRegression(solver='liblinear', class_weight='balanced'))
    ])

    pipeline.fit(train_df['clean_text'], y_train_axis)

    # Dự đoán trên tập Test cố định
    y_pred_axis = pipeline.predict(test_df['clean_text'])

    # Chuyển đổi 0/1 thành chữ cái (Ví dụ: 0 -> I, 1 -> E)
    # Lưu ý: axis[0] là char0 (I), axis[1] là char1 (E) không đúng logic index string
    # Logic đúng: char0 ứng với nhãn 0, char1 ứng với nhãn 1
    predicted_chars = [char1 if pred == 1 else char0 for pred in y_pred_axis]

    # Cộng dồn vào chuỗi kết quả (Ví dụ: "" + "I" -> "I" + "N" -> "IN"...)
    test_df['predicted_type'] += predicted_chars

# 3. So sánh kết quả
# Kiểm tra xem Type gốc có trùng khít với Type dự đoán không
test_df['is_perfect'] = test_df['type'] == test_df['predicted_type']

# 4. Lọc ra các mẫu đúng
perfect_samples = test_df[test_df['is_perfect'] == True]

print(f"\n✅ Đã xong! Tìm thấy {len(perfect_samples)} mẫu đúng cả 4 trục trên tổng {len(test_df)} mẫu test.")
print("="*60)

# [BƯỚC 6] XUẤT RA 5 VÍ DỤ ĐỂ DEMO
# Lấy ngẫu nhiên 5 mẫu để demo
if len(perfect_samples) > 0:
    demo_samples = perfect_samples.sample(min(5, len(perfect_samples)), random_state=1)

    for idx, row in demo_samples.iterrows():
        print(f"\n🎯 VÍ DỤ DEMO #{idx}")
        print(f"🔸 MBTI Thực tế: {row['type']}")
        print(f"🔹 MBTI Dự đoán: {row['predicted_type']}")
        # Lấy 300 ký tự đầu để hiển thị cho đỡ dài
        display_text = row['posts'][:300].replace('\n', ' ')
        print(f"💬 Nội dung (Trích đoạn): \"{display_text}...\"")
        print("-" * 30)
else:
    print("Rất tiếc, không có mẫu nào đúng cả 4 trục. Hãy thử tăng lượng dữ liệu hoặc tinh chỉnh model.")


💎 ĐANG TÌM CÁC MẪU ĐÚNG TUYỆT ĐỐI (4/4)...
 -> Đang train lại & dự đoán trục IE...
 -> Đang train lại & dự đoán trục NS...
 -> Đang train lại & dự đoán trục TF...
 -> Đang train lại & dự đoán trục JP...

✅ Đã xong! Tìm thấy 12554 mẫu đúng cả 4 trục trên tổng 21214 mẫu test.

🎯 VÍ DỤ DEMO #24163
🔸 MBTI Thực tế: INTP
🔹 MBTI Dự đoán: INTP
💬 Nội dung (Trích đoạn): "lethal amount violence stick think average people read verse tell kill gay go meh say holy book guess grab gun go kill gay neighbour hold beer think people mental heah issue read find way justify action importantly think moderate believer accept fact thing part religion enable violence legitimise id..."
------------------------------

🎯 VÍ DỤ DEMO #30820
🔸 MBTI Thực tế: INTP
🔹 MBTI Dự đoán: INTP
💬 Nội dung (Trích đoạn): "prove islam evil prove christianity judaism evil want quote creepy stuff bible cause go road fell like already death mac option e option make float accent next letter type fall accent use feature lot hey big av

In [ ]:
# [BƯỚC 8] XUẤT DỮ LIỆU RA FILE TEXT
from google.colab import files # Chỉ cần dòng này nếu chạy trên Colab

# 1. Chọn số lượng mẫu muốn xuất (Ví dụ lấy 50 mẫu để gửi cho phong phú)
# Nếu muốn lấy hết thì dùng: export_df = perfect_samples
num_samples = 50
if len(perfect_samples) < num_samples:
    num_samples = len(perfect_samples)

export_df = perfect_samples.sample(num_samples, random_state=42)

# 2. Tên file muốn lưu
filename = "demo_mbti_result.txt"

# 3. Ghi vào file
try:
    with open(filename, "w", encoding="utf-8") as f:
        f.write(f"DANH SÁCH {num_samples} MẪU DỰ ĐOÁN ĐÚNG TUYỆT ĐỐI (4 TRỤC MBTI)\n")
        f.write("="*60 + "\n\n")

        for idx, row in export_df.iterrows():
            f.write(f"📌 MẪU SỐ: {idx}\n")
            f.write(f"🏷️ LOẠI MBTI (GỐC & DỰ ĐOÁN): {row['type']}\n")
            f.write(f"💬 NỘI DUNG:\n")
            # Ghi nội dung gốc (chưa clean) để người đọc dễ hiểu hơn
            # Hoặc dùng row['clean_text'] nếu muốn gửi bản đã xử lý
            content = row['posts'].replace('\n', ' ')
            f.write(f"{content}\n")
            f.write("-" * 60 + "\n\n")

    print(f"✅ Đã tạo xong file '{filename}' thành công!")

    # 4. Tải file xuống máy (Dành cho Google Colab)
    print("⬇️ Đang tải file xuống máy tính...")
    files.download(filename)

except Exception as e:
    print(f"❌ Có lỗi khi ghi file: {e}")

✅ Đã tạo xong file 'demo_mbti_result.txt' thành công!
⬇️ Đang tải file xuống máy tính...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>